# Herogod — Qwen Image Edit Studio with a Hugging Face token (Colab)

Same model as the default notebook, but the whole runtime authenticates with **your own
Hugging Face token**:

* `Qwen/Qwen-Image-Edit-2511` (`QwenImageEditPlusPipeline`)
* `ScottzillaSystems/qwen-image-edit-plus-nsfw-lora` — **MCNL v1** adapter, 590 MB

**Why use a token?**
* The LoRA repo carries a *Not-For-All-Audiences* gate — a logged-in token that has accepted the
  page's conditions is the reliable way to fetch it.
* Authenticated downloads get much higher rate limits (plus `hf_transfer` for speed).
* Gated/private mirrors and your own forks work without code changes.

**Before you start**
1. Create a token at <https://huggingface.co/settings/tokens> (a **Read** token is enough;
   fine-grained tokens need *read access to public gated repos*).
2. While logged in, open both model pages and accept their conditions
   ([base](https://huggingface.co/Qwen/Qwen-Image-Edit-2511),
   [LoRA](https://huggingface.co/ScottzillaSystems/qwen-image-edit-plus-nsfw-lora)).
3. Add it to Colab Secrets as `HF_TOKEN` (key icon in the left sidebar, notebook access enabled),
   or type it into the hidden prompt in cell 2.
4. Runtime → Change runtime type → GPU. L4/A100 is comfortable; a free T4 works with fp16 and a
   smaller pixel budget.

The token stays in this runtime's memory and environment. It is never printed, written to disk,
or sent to the studio frontend.

**Adults only.** Open weights with an adult-content adapter. Never create sexual content involving
minors, non-consensual imagery, harassment or fraud. You are responsible for what you generate.


## 1 · Token and runtime settings

In [ ]:
# @title Hugging Face token & runtime { display-mode: "form" }
TOKEN_SOURCE = "Colab Secrets (HF_TOKEN)" # @param ["Colab Secrets (HF_TOKEN)", "Type it now (hidden input)", "Already exported in this runtime"]
ENABLE_HF_TRANSFER = True # @param {type:"boolean"}
BASE_MODEL = "Qwen/Qwen-Image-Edit-2511" # @param ["Qwen/Qwen-Image-Edit-2511"] {allow-input: true}
LORA_REPO = "ScottzillaSystems/qwen-image-edit-plus-nsfw-lora" # @param {type:"string"}
DTYPE = "auto" # @param ["auto", "bf16", "fp16", "fp32"]
QUANT = "none" # @param ["none", "bnb4"]
OFFLOAD = "none" # @param ["none", "model", "sequential"]
MAX_PIXELS = 1048576 # @param {type:"integer"}
DEFAULT_STEPS = 20 # @param {type:"integer"}
DOWNLOAD_NOW = True # @param {type:"boolean"}
PUBLIC_TUNNEL = True # @param {type:"boolean"}
RUN_SMOKE_TEST = True # @param {type:"boolean"}

import getpass
import hashlib
import os
import secrets


def resolve_token(source: str) -> str:
    """Read the token from Colab Secrets, a hidden prompt, or the environment."""
    if source.startswith("Colab"):
        try:
            from google.colab import userdata  # type: ignore

            return (userdata.get("HF_TOKEN") or "").strip()
        except Exception as error:  # secret missing or notebook access disabled
            print("Could not read the Colab secret HF_TOKEN:", error)
            print("Add it with the key icon in the left sidebar, enable notebook access, then re-run this cell.")
            print("Or switch TOKEN_SOURCE to 'Type it now (hidden input)'.")
            return ""
    if source.startswith("Type"):
        return getpass.getpass("Paste your Hugging Face token (input is hidden): ").strip()
    return (os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN") or "").strip()


HF_TOKEN = resolve_token(TOKEN_SOURCE)
if not HF_TOKEN:
    raise SystemExit(
        "A Hugging Face token is required for this notebook.\n"
        "  1. Create a Read token: https://huggingface.co/settings/tokens\n"
        "  2. Add it as the Colab secret HF_TOKEN, or choose the hidden-input source above.\n"
        "  3. Accept the conditions on both model pages with the same account."
    )
if not HF_TOKEN.startswith("hf_"):
    print("Warning: the token does not start with 'hf_'. Double-check that you pasted the whole token.")

# Never print the token: a short hash is enough to confirm which one is loaded.
fingerprint = hashlib.sha256(HF_TOKEN.encode()).hexdigest()[:8]
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" if ENABLE_HF_TRANSFER else "0"
os.environ.update({
    "HEROGOD_BASE_MODEL": BASE_MODEL,
    "HEROGOD_LORA_REPO": LORA_REPO,
    "HEROGOD_DTYPE": DTYPE,
    "HEROGOD_QUANT": QUANT,
    "HEROGOD_OFFLOAD": OFFLOAD,
    "HEROGOD_MAX_PIXELS": str(MAX_PIXELS),
    "HEROGOD_STEPS": str(DEFAULT_STEPS),
    "HEROGOD_VAE_TILING": "1",
    "HEROGOD_MODELS_DIR": "/content/models",
    # Fresh each run; printed at the end so the studio can be unlocked.
    "HEROGOD_API_TOKEN": secrets.token_urlsafe(24),
})

print(f"HF token loaded (source: {TOKEN_SOURCE}, fingerprint: {fingerprint})")
print(f"hf_transfer: {'on' if ENABLE_HF_TRANSFER else 'off'}")
print(f"Model: {BASE_MODEL}  +  LoRA {LORA_REPO}")
try:
    import torch

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} · {props.total_memory / 1024**3:.1f} GB VRAM")
        if props.total_memory < 20e9:
            print("Tight VRAM: consider QUANT='bnb4' and MAX_PIXELS=786432.")
    else:
        print("No GPU attached. Edits will be extremely slow.")
except Exception:
    print("PyTorch is not installed yet; the install cell will handle it.")


## 2 · Get the code

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/tao-mihiranga/Herogod.git"
BRANCH = ""  # leave empty for the default branch

PROJECT_DIR = Path("/content/Herogod")
if (Path.cwd() / "server.py").is_file():
    PROJECT_DIR = Path.cwd()
elif not (PROJECT_DIR / "server.py").is_file():
    cmd = ["git", "clone", "--depth", "1"] + (["--branch", BRANCH] if BRANCH else []) + [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)


## 3 · Install

In [ ]:
%%bash
set -e
echo "Installing server + diffusion stack (quiet, a few minutes)…"
pip -q install -r requirements.txt
pip -q install -r requirements-local.txt
# used by the smoke-test cell to talk to the local API
pip -q install requests
if [ "$HEROGOD_QUANT" = "bnb4" ]; then
  pip -q install -r requirements-quant.txt
fi
if [ "$HF_HUB_ENABLE_HF_TRANSFER" = "1" ]; then
  pip -q install hf_transfer
fi
python - <<'EOF'
import importlib
for name in ("torch", "diffusers", "transformers", "accelerate", "fastapi", "huggingface_hub"):
    module = importlib.import_module(name)
    print(f"{name:16} {getattr(module, '__version__', '?')}")
EOF


## 4 · Verify the token and repo access

This costs nothing and catches the two problems that otherwise surface 40 GB into a download:
a rejected token, or a model gate that has not been accepted.

* `Authenticated as …` comes from `whoami` — it confirms the token is live.
* `[ok] base model … (public)` and `[ok] LoRA … (gated)` confirm both repos are reachable with it.
* A `[FAIL]` line tells you exactly what to fix; nothing is downloaded until this passes.

In [ ]:
import os
import subprocess
import sys

from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

try:
    who = HfApi().whoami(token=HF_TOKEN)
except HfHubHTTPError as error:
    status = getattr(getattr(error, "response", None), "status_code", None)
    if status in (401, 403):
        raise SystemExit(
            "The token was rejected by Hugging Face.\n"
            f"  Server said: {error}\n"
            "  Create a Read token at https://huggingface.co/settings/tokens and re-run cell 1."
        ) from error
    raise SystemExit(f"Hugging Face rejected the request (HTTP {status}): {error}") from error
except Exception as error:  # network / DNS problems inside the runtime
    raise SystemExit(
        "Could not reach huggingface.co from this runtime.\n"
        f"  {type(error).__name__}: {str(error)[:300]}\n"
        "  Check Runtime → Connect, then re-run this cell."
    ) from error

print("Authenticated as:", who.get("name"), f"({who.get('type', 'user')})")
auth = who.get("auth", {})
access_token = auth.get("accessToken", {}) if isinstance(auth, dict) else {}
if access_token.get("role"):
    print("Token role     :", access_token.get("role"))

check = subprocess.run(
    [sys.executable, "scripts/download_models.py", "--check-access"],
    env={**os.environ},
)
if check.returncode != 0:
    raise SystemExit(
        "Access check failed (see the messages above).\n"
        "  Common fixes: accept the conditions on both model pages while logged in;\n"
        "  for fine-grained tokens, allow 'Read access to public gated repos';\n"
        "  make sure the token belongs to the account that accepted the gates."
    )
print("\nAccess check passed — safe to download.")


## 5 · Download the weights into Colab

~40 GB for the base model in bf16, 590 MB for the LoRA. The token is read from the environment,
so it never appears on the command line or in the log. Set `DOWNLOAD_NOW = False` in cell 1 to skip
this and let the server fetch lazily on the first edit instead.

Downloads resume if they are interrupted; re-run the cell after a disconnect.

In [ ]:
if DOWNLOAD_NOW:
    import os
    import subprocess
    import sys

    result = subprocess.run(
        [sys.executable, "scripts/download_models.py", "--dest", "/content/models"],
        env={**os.environ},
    )
    if result.returncode != 0:
        print("\nDownload failed — see the hint above. Nothing is started; fix it and re-run this cell.")
    else:
        base_dir = f"/content/models/{BASE_MODEL.replace('/', '--')}"
        if os.path.isdir(base_dir):
            os.environ["HEROGOD_BASE_MODEL_PATH"] = base_dir
            print("Base model path:", base_dir)
        lora_dir = f"/content/models/{LORA_REPO.replace('/', '--')}"
        if os.path.isdir(lora_dir):
            os.environ["HEROGOD_LORA_REPO"] = lora_dir
            print("LoRA path      :", lora_dir)
else:
    print("Skipped. The server will download with your HF token on the first generation.")


## 6 · Start the studio server
The server inherits `HF_TOKEN`, so any lazy download (base model or LoRA) is authenticated too.

In [ ]:
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

LOG = Path("/tmp/herogod-server.log")
server_log = LOG.open("w")
server_process = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError("Server exited:\n" + LOG.read_text()[-4000:])
    try:
        with socket.create_connection(("127.0.0.1", 8000), timeout=1):
            break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError("Server did not open port 8000:\n" + LOG.read_text()[-4000:])

print("Herogod server is running on port 8000. Log:", LOG)


## 7 · Open the studio

`PUBLIC_TUNNEL = True` gives a temporary Cloudflare URL you can open anywhere (protected by the
access token). Set it to `False` to use Colab's private proxy instead — nothing is exposed publicly.

In [ ]:
import os
import platform
import re
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

TOKEN = os.environ["HEROGOD_API_TOKEN"]

if not PUBLIC_TUNNEL:
    from google.colab.output import eval_js

    url = eval_js("google.colab.kernel.proxyPort(8000)")
    print("Studio URL  :", url)
    print("Access token:", TOKEN)
else:
    if platform.machine() not in ("x86_64", "AMD64"):
        raise RuntimeError("The cloudflared launcher expects a 64-bit Intel/AMD runtime. Use PUBLIC_TUNNEL=False.")
    cloudflared = Path("/tmp/cloudflared")
    if not cloudflared.exists():
        print("Downloading cloudflared…")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            cloudflared,
        )
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

    tunnel_log_path = Path("/tmp/herogod-tunnel.log")
    tunnel_log = tunnel_log_path.open("w")
    tunnel_process = subprocess.Popen(
        [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
    )

    public_url = None
    for _ in range(120):
        if tunnel_process.poll() is not None:
            break
        match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", tunnel_log_path.read_text())
        if match:
            public_url = match.group(0)
            break
        time.sleep(1)

    if not public_url:
        raise RuntimeError("Tunnel failed:\n" + tunnel_log_path.read_text()[-3000:])

    print("\n" + "─" * 72)
    print("HEROGOD IS READY")
    print("Studio:        ", public_url)
    print("One-click link:", f"{public_url}/?token={TOKEN}")
    print("Access token:  ", TOKEN)
    print("─" * 72)
    print("The HF token is NOT this token: the studio never sees or needs it.")


## 8 · Smoke test (proves the token actually works)

Runs one real edit through the API with a generated sample image. This is the step that exercises
the authenticated weights end to end: pipeline build, LoRA load, generation, PNG download.
The first run loads the model, so expect several minutes on a T4.

In [ ]:
if RUN_SMOKE_TEST:
    import io
    import json
    import time
    import urllib.request

    import requests
    from PIL import Image, ImageDraw

    canvas = Image.new("RGB", (768, 768), (24, 26, 38))
    draw = ImageDraw.Draw(canvas)
    draw.ellipse((190, 150, 578, 538), fill=(236, 176, 148))
    draw.rectangle((250, 560, 518, 768), fill=(70, 96, 160))
    draw.text((330, 700), "smoke test", fill=(255, 255, 255))
    buffer = io.BytesIO()
    canvas.save(buffer, format="PNG")

    headers = {"X-Herogod-Token": TOKEN}
    started = time.time()
    print("Queuing an edit (model loads on the first request)…")
    response = requests.post(
        "http://127.0.0.1:8000/api/jobs",
        headers=headers,
        files={"image": ("smoke.png", buffer.getvalue(), "image/png")},
        data={
            "prompt": "add a soft warm studio light and a plain grey backdrop, keep the subject unchanged",
            "steps": "8",
            "true_cfg": "3.0",
            "lora_scale": "0.0",
        },
        timeout=300,
    )
    if response.status_code != 202:
        raise SystemExit(f"Could not queue the job: {response.status_code} {response.text[:400]}")
    job = response.json()
    print("Job:", job["id"], "| output:", job["settings"]["size"])

    last = ""
    while True:
        if time.time() - started > 2400:
            raise SystemExit("Timed out after 40 minutes. Check /tmp/herogod-server.log.")
        status = requests.get(f"http://127.0.0.1:8000/api/jobs/{job['id']}", headers=headers, timeout=60).json()
        line = f"{status['status']} {status['step']}/{status['steps']}"
        if line != last:
            print(f"  {line}", end="\r")
            last = line
        if status["status"] in ("done", "error", "cancelled"):
            break
        time.sleep(2)

    print()
    if status["status"] != "done":
        raise SystemExit(f"Smoke test failed: {status.get('error') or status['status']}")

    image_bytes = requests.get(
        f"http://127.0.0.1:8000/api/jobs/{job['id']}/image", headers=headers, timeout=120
    ).content
    out = "/content/herogod-smoke-test.png"
    open(out, "wb").write(image_bytes)
    print(f"OK — seed {status['seed']}, {status['elapsed']}s, device {status['meta'].get('device')}")
    print("Saved:", out)
    print("The token, the gated LoRA and the pipeline all work.")
else:
    print("Smoke test disabled. The studio is still ready at the URL from the previous cell.")


## Troubleshooting

| Message | What to do |
| --- | --- |
| `Token was rejected` (401) | Recreate the token at <https://huggingface.co/settings/tokens> and re-run cell 1. Hidden input avoids copy/paste whitespace issues. |
| `Access to model … is restricted` / `[FAIL] LoRA … gated` | Log in on the Hub with the token's account, open each model page and accept the conditions (the LoRA has an age/NSFW gate). Then re-run cell 4. |
| `403 Forbidden` on the LoRA only | Fine-grained token without *public gated repos* read access, or the token belongs to a different account than the one that accepted the gate. |
| `429` / slow downloads | Set `ENABLE_HF_TRANSFER = True` (installs `hf_transfer`) and keep the token loaded — authenticated requests have higher limits. |
| `Could not read the Colab secret` | Left sidebar key icon → add `HF_TOKEN` → enable notebook access. Or use `TOKEN_SOURCE = "Type it now (hidden input)"`. |
| `CUDA out of memory` | Lower `MAX_PIXELS` (786432 or 589520), set `QUANT='bnb4'` or `OFFLOAD='model'`, restart the runtime. |
| Slow on a T4 | `DTYPE='fp16'`, `MAX_PIXELS=786432`, 8–12 steps. |
| `Cannot reach the backend` in the studio | Re-run cell 6; check `/tmp/herogod-server.log`. |
| Tunnel URL dead | Re-run cell 7; quick tunnels are temporary. |
| Notebook disconnected | The runtime (and the model in VRAM) is gone. Reconnect and re-run from cell 1; `/content/models` survives while the VM lasts. |

**Token hygiene:** this notebook only reads the token, never prints or stores it. `HF_TOKEN` lives in
the runtime's environment — a shared or borrowed runtime means someone else can read your secrets.
Revoke the token when you are done experimenting.

**Licences:** Qwen-Image-Edit-2511 is Apache-2.0; the MCNL LoRA is OpenRAIL++. Read both model cards.


## Stop

In [ ]:
import os

for name in ("tunnel_process", "server_process"):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        print("Stopped", name)

# Drop the token from this process before leaving the notebook idle.
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
print("HF token removed from the environment.")
